In [ ]:
import argparse
import logging
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import random
from transformers import AutoTokenizer

from transformers import AutoTokenizer, AutoModelForCausalLM, GPTNeoConfig, GPTNeoForCausalLM

cfg_param = "8M"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
epochs = 1
seed = 3407
batch_size = 64
window_size = 256
lr = 1e-3

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
    

# tokenizer = AutoTokenizer.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")
# model = AutoModelForCausalLM.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")

# # Initializing a model (with random weights) from the EleutherAI/gpt-neo-1.3B style configuration
# model = GPTNeoForCausalLM(model.config)

# num_params = sum(p.numel() for p in model.parameters())
# print(f"Number of parameters in model: {num_params}")

# Load HuggingFace token from .env file
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import HfApi, login
import json

# Login to HuggingFace
hf_token = os.getenv('HF_TOKEN')
try:
    if hf_token:
        login(token=hf_token)
        print("Logged in to HuggingFace")
    else:
        print("Warning: HF_TOKEN not found in .env file")
except Exception as e:
    print(e)

# Set your HuggingFace username/organization
HF_USERNAME = os.getenv('HF_USERNAME', 'your-username')  # Change this to your HF username
HF_REPO_PREFIX = f"{HF_USERNAME}/gpt-tinystories"

/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================
# TEXT GENERATION / INFERENCE
# ============================================

def load_model_for_inference(repo_name=None, checkpoint_step=None, device='cuda'):
    """
    Load a trained model from HuggingFace for text generation
    
    Args:
        repo_name: Full HuggingFace repo name (e.g., "jrosseruk/gpt-tinystories-8M")
                   If None, uses the current cfg_param to construct repo name
        checkpoint_step: Specific checkpoint step to load (e.g., 1000, 2000)
                        If None, loads the latest model from main branch
        device: Device to load model on ('cuda' or 'cpu')
    
    Returns:
        model: The loaded model
        tokenizer: The tokenizer
    """
    if repo_name is None:
        repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    
    # Determine subfolder if checkpoint_step is specified
    subfolder = None
    if checkpoint_step is not None:
        subfolder = f"checkpoint-{checkpoint_step}"
        print(f"Loading model from {repo_name}/{subfolder}...")
    else:
        print(f"Loading latest model from {repo_name}...")
    
    try:
        from huggingface_hub import repo_exists, list_repo_files
        
        # Check if repo exists
        if not repo_exists(repo_name):
            print(f"Error: Repository {repo_name} does not exist on HuggingFace Hub")
            print(f"Please check the repository name or train a model first")
            return None, None
        
        # If loading a specific checkpoint, verify it exists
        if subfolder is not None:
            try:
                files = list_repo_files(repo_id=repo_name)
                checkpoint_files = [f for f in files if f.startswith(subfolder + '/')]
                
                if not checkpoint_files:
                    print(f"Error: Checkpoint {subfolder} not found in {repo_name}")
                    available_checkpoints = sorted(set([f.split('/')[0] for f in files if f.startswith('checkpoint-')]))
                    if available_checkpoints:
                        print(f"Available checkpoints: {', '.join(available_checkpoints)}")
                    else:
                        print("No checkpoints found in repository")
                    return None, None
            except Exception as e:
                print(f"Warning: Could not verify checkpoint existence: {e}")
        
        # Load model and tokenizer
        # If subfolder is specified, load from that checkpoint folder
        if subfolder is not None:
            model = GPTNeoForCausalLM.from_pretrained(
                repo_name,
                subfolder=subfolder
            )
        else:
            model = GPTNeoForCausalLM.from_pretrained(repo_name)
        
        tokenizer = AutoTokenizer.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")
        tokenizer.pad_token = tokenizer.eos_token
        
        # Move to device and set to eval mode
        model = model.to(device)
        model.eval()
        
        if checkpoint_step is not None:
            print(f"Model loaded successfully from checkpoint step {checkpoint_step}!")
        else:
            print(f"Model loaded successfully!")
        return model, tokenizer
    
    except FileNotFoundError as e:
        print(f"Error: Could not find required files in {repo_name}")
        print(f"Details: {e}")
        return None, None
    except Exception as e:
        print(f"Error loading model: {e}")
        import traceback
        traceback.print_exc()
        return None, None


def generate_text(model, tokenizer, prompt, max_length=100, temperature=0.8, top_k=50, top_p=0.95, num_return_sequences=1, device='cuda'):
    """
    Generate text completion from a prompt
    
    Args:
        model: The trained model
        tokenizer: The tokenizer
        prompt: Text prompt to complete
        max_length: Maximum length of generated text (in tokens)
        temperature: Sampling temperature (higher = more random, lower = more deterministic)
        top_k: Keep only top k tokens with highest probability (0 = disabled)
        top_p: Nucleus sampling - keep top tokens with cumulative probability >= top_p
        num_return_sequences: Number of different completions to generate
        device: Device model is on
    
    Returns:
        List of generated text completions
    """
    model.eval()
    
    # Encode the prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        # Generate
        output_sequences = model.generate(
            input_ids=input_ids,
            max_length=max_length + len(input_ids[0]),
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            num_return_sequences=num_return_sequences,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode the generated sequences
    generated_texts = []
    for sequence in output_sequences:
        text = tokenizer.decode(sequence, skip_special_tokens=True)
        generated_texts.append(text)
    
    return generated_texts


# # Load from checkpoint 1000
# model, tokenizer = load_model_for_inference(checkpoint_step=33000)

# # Or load the latest model:
# # model, tokenizer = load_model_for_inference()

# if model is not None:
#     # Define a prompt
#     prompt = "Victorious"
    
#     print(f"Prompt: {prompt}")
#     print("=" * 50)
    
#     # Generate completions
#     completions = generate_text(
#         model, 
#         tokenizer, 
#         prompt, 
#         max_length=150,
#         temperature=0.8,
#         num_return_sequences=3  # Generate 3 different completions
#     )
    
#     # Print results
#     for i, completion in enumerate(completions, 1):
#         print(f"\nCompletion {i}:")
#         print(completion)
#         print("=" * 50)


# print("Text generation functions loaded. Uncomment the example usage block to test!")

In [3]:

# Apply kronfluence patches before importing
from infusion.kronfluence_patches import apply_patches
apply_patches()

# Now import kronfluence normally
import sys
sys.path.append("")
sys.path.append("kronfluence")
sys.path.append("kronfluence/kronfluence")
from kronfluence.analyzer import Analyzer, prepare_model
from kronfluence.arguments import FactorArguments, ScoreArguments
from kronfluence.task import Task
from kronfluence.utils.dataset import DataLoaderKwargs


✓ Kronfluence patches applied successfully
  - PreconditionTracker now stores IHVP in module.storage['inverse_hessian_vector_product']


In [4]:
# Helper function to load and inspect saved training data from a checkpoint
def load_checkpoint_data(checkpoint_step):
    """
    Load the training/validation data used for a specific checkpoint
    Args:
        checkpoint_step: The checkpoint step number (e.g., 1000, 2000)
    Returns:
        Dictionary with train_data and val_data
    """
    from huggingface_hub import hf_hub_download, repo_exists, list_repo_files
    import json
    
    repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    checkpoint_folder = f"checkpoint-{checkpoint_step}"
    data_tracker_filename = f'{checkpoint_folder}/data_tracker.json'
    
    try:
        # Check if repo exists
        if not repo_exists(repo_name):
            print(f"Error: Repository {repo_name} does not exist on HuggingFace Hub")
            return None
        
        # Check if checkpoint exists
        files = list_repo_files(repo_id=repo_name)
        checkpoint_files = [f for f in files if f.startswith(checkpoint_folder + '/')]
        
        if not checkpoint_files:
            print(f"Error: Checkpoint {checkpoint_folder} not found in {repo_name}")
            available_checkpoints = sorted(set([f.split('/')[0] for f in files if f.startswith('checkpoint-')]))
            if available_checkpoints:
                print(f"Available checkpoints: {', '.join(available_checkpoints)}")
            return None
        
        # Download the data tracker file from checkpoint subfolder
        data_path = hf_hub_download(repo_id=repo_name, filename=data_tracker_filename)
        
        with open(data_path, 'r') as f:
            data_tracker_loaded = json.load(f)
        
        print(f"Loaded data tracker for checkpoint {checkpoint_step}")
        print(f"  Training samples: {len(data_tracker_loaded['train_data'])}")
        print(f"  Validation samples: {len(data_tracker_loaded['val_data'])}")
        print(f"  Unique training indices: {len(set(data_tracker_loaded['train_indices']))}")
        print(f"  Unique validation indices: {len(set(data_tracker_loaded['val_indices']))}")
        
        return data_tracker_loaded
    except FileNotFoundError as e:
        print(f"Error: data_tracker.json not found in checkpoint {checkpoint_folder}")
        print(f"Details: {e}")
        return None
    except Exception as e:
        print(f"Error loading checkpoint data: {e}")
        import traceback
        traceback.print_exc()
        return None

# Example usage (uncomment to use):
data = load_checkpoint_data(32000)
if data:
    # Access first training sample
    print("First training sample:", data['train_data'][0])
    
    # Get all training texts
    train_texts = [sample['text'] for sample in data['train_data']]
    
    # Verify reproducibility - check if indices match expected order
    print("Training indices:", data['train_indices'][:10])

Loaded data tracker for checkpoint 32000
  Training samples: 64000
  Validation samples: 12800
  Unique training indices: 64000
  Unique validation indices: 10140
First training sample: {'index': 473652, 'text': 'Tom and Lily were playing in the park with their kites. They liked to see their kites fly high in the sky. They tied the strings to their hands so they would not lose their kites.\n\nBut then, they saw dark clouds and heard loud noises. It was a storm! They were scared and ran to their mom, who was sitting on a bench.\n\n"Mom, mom, there is a storm! Can we go home?" they cried.\n\nMom looked at the sky and said, "Yes, we should go home. It is not safe to play outside when there is a storm. Come on, let\'s pack our things."\n\nTom and Lily grabbed their kites and followed their mom. But the wind was very strong and it pulled their kites. They felt dizzy and fell down.\n\n"Ouch, that hurts!" Tom said.\n\n"Are you okay?" Lily asked.\n\n"I think so. But my kite is gone. The wind t

In [6]:
from typing import Dict, List

import torch
import torch.nn.functional as F
from torch import nn

from kronfluence.task import Task

BATCH_TYPE = Dict[str, torch.Tensor]


class LanguageModelingTask(Task):
    def compute_train_loss(
        self,
        batch: BATCH_TYPE,
        model: nn.Module,
        sample: bool = False,
    ) -> torch.Tensor:
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        logits = logits[..., :-1, :].contiguous()
        logits = logits.view(-1, logits.size(-1))
        labels = batch["labels"][..., 1:].contiguous()
        if not sample:
            summed_loss = F.cross_entropy(logits, labels.view(-1), reduction="sum", ignore_index=-100)
        else:
            with torch.no_grad():
                probs = torch.nn.functional.softmax(logits.detach(), dim=-1)
                sampled_labels = torch.multinomial(
                    probs,
                    num_samples=1,
                ).flatten()
                masks = labels.view(-1) == -100
                sampled_labels[masks] = -100
            summed_loss = F.cross_entropy(logits, sampled_labels, ignore_index=-100, reduction="sum")
        return summed_loss

    def compute_measurement(
        self,
        batch: BATCH_TYPE,
        model: nn.Module,
    ) -> torch.Tensor:
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        shift_labels = batch["labels"][..., 1:].contiguous().view(-1)
        logits = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
        return F.cross_entropy(logits, shift_labels, ignore_index=-100, reduction="sum")

    def get_influence_tracked_modules(self) -> List[str]:
        total_modules = []

        # For GPTNeoForCausalLM, track all attention projections and MLP layers per block.
        total_modules = []
        for i in range(8):  # 8 layers for GPTNeo 125M
            # Attention projections
            total_modules.append(f"transformer.h.{i}.attn.attention.q_proj")
            total_modules.append(f"transformer.h.{i}.attn.attention.k_proj")
            total_modules.append(f"transformer.h.{i}.attn.attention.v_proj")
            total_modules.append(f"transformer.h.{i}.attn.attention.out_proj")
            # MLP projections
            total_modules.append(f"transformer.h.{i}.mlp.c_fc")
            total_modules.append(f"transformer.h.{i}.mlp.c_proj")

        return total_modules

    def get_attention_mask(self, batch: BATCH_TYPE) -> torch.Tensor:
        return batch["attention_mask"]

In [ ]:
# from examples.openwebtext.task import LanguageModelingTask

import argparse
from transformers import default_data_collator
from kronfluence.utils.common.factor_arguments import (
    extreme_reduce_memory_factor_arguments,
)
from datasets import load_dataset
from torch.utils.data import Dataset

# Create argument parser and parse arguments
parser = argparse.ArgumentParser(description="GPT-Neo Infusion Jupyter notebook arguments")
parser.add_argument('--damping', type=float, default=1e-3, help="Damping factor for influence computation")
args, _ = parser.parse_known_args()


# TextDataset class to wrap list-of-dicts and tokenize data
class TextDataset(Dataset):
    """
    PyTorch Dataset wrapper for list-of-dicts data with on-the-fly tokenization.
    Converts raw text to tokenized format required by kronfluence.
    """
    def __init__(self, data_list, tokenizer, max_length):
        """
        Args:
            data_list: List of dicts with 'text' key containing raw text
            tokenizer: HuggingFace tokenizer
            max_length: Maximum sequence length for tokenization
        """
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        text = self.data[idx]['text']
        
        # Tokenize the text
        tokenized = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',  # Pad all sequences to max_length for batching
            return_tensors='pt'
        )
        
        # Extract and squeeze (remove batch dimension)
        input_ids = tokenized['input_ids'].squeeze(0)
        attention_mask = tokenized['attention_mask'].squeeze(0)
        
        # Create labels (copy of input_ids with padding tokens set to -100)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }


ckpt = 33000
eval_dataset = load_checkpoint_data(ckpt-1000)

# Load model and tokenizer first (needed for TextDataset)
model, tokenizer = load_model_for_inference(checkpoint_step=ckpt)
model = model.eval()

# Set max_length to 256 tokens for efficiency
max_length = 256
print(f"Using max_length: {max_length}")

#######################################
# WRAP DATASETS IN TEXTDATASET FOR PROPER TOKENIZATION
#######################################
# Wrap eval datasets with TextDataset for proper tokenization
eval_train_dataset = TextDataset(eval_dataset["train_data"], tokenizer, max_length)
valid_train_dataset = TextDataset(eval_dataset["val_data"], tokenizer, max_length)
print(f"Wrapped eval_train_dataset: {len(eval_train_dataset)} samples")
print(f"Wrapped valid_train_dataset: {len(valid_train_dataset)} samples")

# Note: We're only using eval_train_dataset and valid_train_dataset from checkpoint data
# No need to load the full TinyStories dataset since we're focused on the last 1000 training steps

# Create task and prepare model
task = LanguageModelingTask()
model = prepare_model(model, task)

# Set up the Analyzer class.
analyzer = Analyzer(
    analysis_name="gpt_neo",
    model=model,
    task=task,
)
# Configure parameters for DataLoader.
dataloader_kwargs = DataLoaderKwargs(num_workers=0, collate_fn=default_data_collator, pin_memory=True)
analyzer.set_dataloader_kwargs(dataloader_kwargs)

factors_name = "ekfac"
factor_args = extreme_reduce_memory_factor_arguments(
    strategy="ekfac", module_partitions=1, dtype=torch.bfloat16
)
factor_args.covariance_module_partitions = 2
factor_args.lambda_module_partitions = 4
factor_args.covariance_data_partitions = 4
factor_args.lambda_data_partitions = 4
analyzer.fit_all_factors(
    factors_name=factors_name,
    dataset=eval_train_dataset,  # Fit on 64K samples from last 1000 training steps (checkpoint 32000-33000)
    per_device_batch_size=4,
    factor_args=factor_args,
    overwrite_output_dir=False,
)




Loaded data tracker for checkpoint 32000
  Training samples: 64000
  Validation samples: 12800
  Unique training indices: 64000
  Unique validation indices: 10140
Loading model from jrosseruk/gpt-tinystories-8M/checkpoint-33000...


/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/factor/covariance.py:200: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)


Model loaded successfully from checkpoint step 33000!
Using max_length: 256
Wrapped eval_train_dataset: 64000 samples
Wrapped valid_train_dataset: 12800 samples


Fitting covariance matrices [4000/4000] 100%|██████████ [time left: 00:00, time spent: 01:58]
Fitting covariance matrices [4000/4000] 100%|██████████ [time left: 00:00, time spent: 01:57]
Fitting covariance matrices [4000/4000] 100%|██████████ [time left: 00:00, time spent: 01:57]
Fitting covariance matrices [4000/4000] 100%|██████████ [time left: 00:00, time spent: 01:59]
Fitting covariance matrices [4000/4000] 100%|██████████ [time left: 00:00, time spent: 01:57]
Fitting covariance matrices [4000/4000] 100%|██████████ [time left: 00:00, time spent: 01:58]
Fitting covariance matrices [4000/4000] 100%|██████████ [time left: 00:00, time spent: 01:57]
Fitting covariance matrices [4000/4000] 100%|██████████ [time left: 00:00, time spent: 01:58]
Performing Eigendecomposition [48/48] 100%|██████████ [time left: 00:00, time spent: 00:00]
/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/factor/eigen.py:398: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `t

Using damping factor: 0.001


/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/score/pairwise.py:206: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)


RuntimeError: No executable batch size found, reached zero.

In [14]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Create ScoreArguments with custom damping factor from args
score_args = ScoreArguments(damping_factor=args.damping)
print(f"Using damping factor: {args.damping}")

query_dataset = TextDataset(eval_dataset["val_data"][:2], tokenizer, max_length)

# Compute pairwise influence scores.
analyzer.compute_pairwise_scores(
    scores_name="ekfac_scores",
    factors_name="ekfac",
    query_dataset=query_dataset,  # 12.8K validation samples from the last 1000 training steps
    train_dataset=eval_train_dataset,  # 64K training samples from the last 1000 training steps
    per_device_query_batch_size=1,
    score_args=score_args,
    overwrite_output_dir=True,
)

Using damping factor: 0.001


Exception ignored in: <function _ConnectionBase.__del__ at 0x400562fba0e0>
Traceback (most recent call last):
  File "/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/multiprocessing/connection.py", line 132, in __del__
Traceback (most recent call last):
  File "/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/multiprocessing/queues.py", line 239, in _feed
    reader_close()
  File "/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/multiprocessing/connection.py", line 177, in close
    self._close()
  File "/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/multiprocessing/connection.py", line 361, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor
    self._close()
  File "/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/multiprocessing/connection.py", line 361, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor
Computing pairwise scores (training gradient) [250/2

In [15]:
scores = analyzer.load_pairwise_scores("ekfac_scores")
print(scores.keys())
print(scores)
print(scores["all_modules"].shape)

dict_keys(['all_modules'])
{'all_modules': tensor([[ 26038.3984, -22959.6348, -37388.1602,  ..., -24792.2188,
         -22256.4043,  33084.5156],
        [-23395.1016,  55270.8438,  75548.5625,  ...,  84956.0859,
          56435.1328,  44597.5273]])}
torch.Size([2, 64000])


In [ ]:
# Display each query passage and its top 20 most influential training texts
import torch

# Get the influence scores (shape: [2, 64000])
influence_scores = scores["all_modules"]

# Process each query passage
for query_idx in range(influence_scores.shape[0]):
    print("=" * 100)
    print(f"QUERY PASSAGE {query_idx + 1}")
    print("=" * 100)
    
    # Display the query passage
    query_text = eval_dataset["val_data"][query_idx]["text"]
    print(f"\n{query_text}\n")
    
    # Get influence scores for this query passage
    passage_scores = influence_scores[query_idx]
    
    # Get indices of top 20 most influential texts (highest scores)
    top_k = 20
    top_scores, top_indices = torch.topk(passage_scores, k=top_k, largest=True)
    
    print("\n" + "-" * 100)
    print(f"TOP {top_k} MOST INFLUENTIAL TRAINING TEXTS (highest influence scores)")
    print("-" * 100)
    
    # Display each influential text with its score
    for rank, (score, train_idx) in enumerate(zip(top_scores, top_indices), 1):
        train_text = eval_dataset["train_data"][train_idx.item()]["text"]
        train_index = eval_dataset["train_data"][train_idx.item()]["index"]
        
        print(f"\n[Rank {rank}] Influence Score: {score.item():.4f} | Training Index: {train_index}")
        print(f"{train_text}")
        print("-" * 80)
    
    print("\n\n")